# Lab 3 — YOLO v1: the building blocks (and TensorBoard)

So far our networks answered *"what is in this image?"* (classification). **Object detection** also asks
*"where?"*: for every object we want a **bounding box** and a **class**.

**YOLO** (*You Only Look Once*) is a *single-shot* detector: instead of classifying thousands of region proposals
(as R-CNN does), a single forward pass of one network looks at the whole image and outputs all the boxes at once.
That's why it is so fast.

In this notebook we build, one at a time, all the pieces that `yolo_voc.py` uses:

1. The dataset: Pascal VOC and its annotations
2. Bounding box formats
3. The S×S grid and the target tensor
4. The output tensor: $S \times S \times (5B + K)$
5. The architecture: backbone (and pretrained weights), 1×1 convolution, fully connected, reshape
6. Intersection over Union (IoU)
7. The YOLO loss
8. From the output to the detections: confidence threshold and Non-Maximum Suppression
9. How detectors are evaluated: mAP
10. Logging the training with TensorBoard

Afterwards, in `yolo_voc.py`, you will implement the network and compose its loss, and then train it.

In [1]:
import os
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torchvision import models, ops
from torchvision.transforms import functional as TF
from torchvision.utils import draw_bounding_boxes

from yolo_voc import DATA_DIR, VOC_CLASSES, download_voc   # we reuse the download code of the exercise

torch.manual_seed(0)

S, B, K = 7, 2, len(VOC_CLASSES)   # grid size, boxes per cell, number of classes
IMG_SIZE = 224
print(f"S={S}, B={B}, K={K}: {K} classes -> {VOC_CLASSES}")

ModuleNotFoundError: No module named 'yolo_voc'

---
## 1. The dataset: Pascal VOC

[Pascal VOC 2007](http://host.robots.ox.ac.uk/pascal/VOC/) is a classic detection benchmark: ~10,000 images (half
for training, half for testing) with ~24,000 annotated objects of 20 classes.

⚠️ The next cell downloads it (~870 MB), only the first time, into `data_dir` (by default `Documents/datasets/VOC`
in your home folder, the same as `yolo_voc.py`; change it if you want). If possible, download it before the lesson
with `python yolo_voc.py --download_only`.

In [ ]:
data_dir = DATA_DIR                    # where Pascal VOC is downloaded: change it if you want
voc_dir = download_voc(data_dir)
print(voc_dir)
print(os.listdir(voc_dir))

Each image `JPEGImages/<id>.jpg` has an annotation `Annotations/<id>.xml` listing the objects, each with its class and
its box as pixel coordinates of the top-left and bottom-right corners:

In [ ]:
image_id = "000012"
with open(os.path.join(voc_dir, "Annotations", f"{image_id}.xml")) as f:
    print(f.read())

Let's read the boxes and draw them with `torchvision.utils.draw_bounding_boxes`, which wants a `uint8` image tensor
and boxes as `[xmin, ymin, xmax, ymax]`.

In [ ]:
def parse_annotation(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        if int(obj.find("difficult").text):          # VOC marks some objects as "difficult": we skip them
            continue
        bb = obj.find("bndbox")
        boxes.append([float(bb.find(tag).text) for tag in ("xmin", "ymin", "xmax", "ymax")])
        labels.append(VOC_CLASSES.index(obj.find("name").text))
    return torch.tensor(boxes).reshape(-1, 4), torch.tensor(labels)


def show(img_uint8, title=None, ax=None):
    ax = ax or plt.gca()
    ax.imshow(img_uint8.permute(1, 2, 0))                 # (C, H, W) -> (H, W, C) (→ tensors §6)
    ax.set_title(title)
    ax.axis("off")


def load_example(image_id):
    image = Image.open(os.path.join(voc_dir, "JPEGImages", f"{image_id}.jpg")).convert("RGB")
    boxes, labels = parse_annotation(os.path.join(voc_dir, "Annotations", f"{image_id}.xml"))
    return TF.pil_to_tensor(image), boxes, labels


img, boxes, labels = load_example("000017")
print("Image:", tuple(img.shape), img.dtype)
print("Boxes (xmin, ymin, xmax, ymax):\n", boxes)
print("Labels:", [VOC_CLASSES[l] for l in labels])

plt.figure(figsize=(6, 5))
show(draw_bounding_boxes(img, boxes, labels=[VOC_CLASSES[l] for l in labels], colors="red", width=3))
plt.show()

---
## 2. Bounding box formats

The same box can be written in different ways:
- **`xyxy`**: corners `[xmin, ymin, xmax, ymax]` — the VOC format, handy for drawing and for the IoU;
- **`cxcywh`**: centre and size `[cx, cy, w, h]` — the format YOLO predicts.

We also **normalize** the coordinates by the image size, so they are in $[0, 1]$. Nice side effect: normalized
boxes do not change when we resize the image to 224×224!

`torchvision.ops.box_convert` converts between the formats.

In [ ]:
H, W = img.shape[1:]
boxes_norm = boxes / torch.tensor([W, H, W, H])                   # xyxy in [0, 1] (broadcasting, → tensors §9)
boxes_cxcywh = ops.box_convert(boxes_norm, in_fmt="xyxy", out_fmt="cxcywh")
print("normalized xyxy:\n", boxes_norm)
print("normalized cxcywh:\n", boxes_cxcywh)

img224 = TF.resize(img, [IMG_SIZE, IMG_SIZE])
boxes224 = boxes_norm * IMG_SIZE                                  # back to pixels, on the resized image
plt.figure(figsize=(4, 4))
show(draw_bounding_boxes(img224, boxes224, colors="red", width=2), "resized to 224x224: same normalized boxes")
plt.show()

---
## 3. The S×S grid and the target tensor

YOLO divides the image into an $S \times S$ grid ($S = 7$). **The cell that contains the centre of an object is
responsible for detecting it.** So each cell predicts:
- the centre of the object **relative to the cell itself**: the cell already knows the centre is inside it, it only
  has to say *where* inside, an offset in $[0, 1]$;
- the size of the box **relative to the whole image** (an object can be much larger than its cell);
- the class.

Let's draw the grid and the centres:

In [ ]:
def show_grid(img224, boxes_cxcywh, ax=None):
    ax = ax or plt.gca()
    ax.imshow(img224.permute(1, 2, 0))
    step = IMG_SIZE / S
    for k in range(1, S):
        ax.axhline(k * step, color="white", lw=0.8)
        ax.axvline(k * step, color="white", lw=0.8)
    for cx, cy, w, h in boxes_cxcywh.tolist():
        col, row = int(cx * S), int(cy * S)
        ax.add_patch(plt.Rectangle((col * step, row * step), step, step, color="yellow", alpha=0.4))
        ax.add_patch(plt.Rectangle(((cx - w / 2) * IMG_SIZE, (cy - h / 2) * IMG_SIZE), w * IMG_SIZE, h * IMG_SIZE,
                                   fill=False, color="red", lw=2))
        ax.plot(cx * IMG_SIZE, cy * IMG_SIZE, "ro")
    ax.axis("off")


plt.figure(figsize=(5, 5))
show_grid(img224, boxes_cxcywh)
plt.title("7x7 grid: the yellow cells are responsible for the objects")
plt.show()

VOC gives us the boxes as $(x_{min}, y_{min}, x_{max}, y_{max})$ in pixels, with no notion of grid. To train the
network we must **convert them into the same encoding the cells predict**: the target for an image is a tensor of
shape $S \times S \times (5 + K)$, and for each responsible cell:

| index | value |
|---|---|
| 0 | 1 = "there is an object here" |
| 1, 2 | $x, y$ = position of the centre **inside the cell**, in $[0, 1]$ |
| 3, 4 | $w, h$ = size of the box **relative to the whole image** |
| 5 … 5+K | one-hot encoding of the class |

All the other cells are zeros. (YOLO v1 can detect at most one object per cell: a known limitation.)

In §4 we will see that the network actually predicts **$B$ boxes per cell**: they are $B$ candidates for the *same*
object (they share the class probabilities, and only the best one is trained on it, §7), not $B$ different objects.

In [ ]:
def encode_target(boxes_cxcywh, labels):
    target = torch.zeros(S, S, 5 + K)
    for (cx, cy, w, h), label in zip(boxes_cxcywh.tolist(), labels.tolist()):
        col = min(int(cx * S), S - 1)
        row = min(int(cy * S), S - 1)
        if target[row, col, 0] == 1:        # cell already taken
            continue
        target[row, col, :5] = torch.tensor([1.0, cx * S - col, cy * S - row, w, h])
        target[row, col, 5 + label] = 1.0
    return target


target = encode_target(boxes_cxcywh, labels)
print("Target shape:", tuple(target.shape))
print("Object mask (target[..., 0]):\n", target[..., 0].int())

for row, col in torch.nonzero(target[..., 0]).tolist():
    cell = target[row, col]
    print(f"cell ({row}, {col}): x={cell[1]:.2f} y={cell[2]:.2f} w={cell[3]:.2f} h={cell[4]:.2f} "
          f"class={VOC_CLASSES[cell[5:].argmax()]}")

---
## 4. The output tensor: $S \times S \times (5B + K)$

The network predicts, for **each cell**:
- $B$ boxes, each with 5 numbers: the confidence $C$ ("there is an object in this box") and $x, y, w, h$;
- $K$ class probabilities, **shared** by the $B$ boxes of the cell.

$x, y, w, h$ have the **same meaning as in the target** (§3): the centre relative to the cell, the size relative to
the whole image. The network learns this encoding because the loss compares its outputs with the target.

With $S = 7$, $B = 2$, $K = 20$: $7 \times 7 \times (5 \cdot 2 + 20) = 7 \times 7 \times 30$. The 30 values of a cell
are ordered as `[C1, x1, y1, w1, h1, C2, x2, y2, w2, h2, p_1, ..., p_20]`.

💡 A practical choice (the same in `yolo_voc.py`): the network outputs raw values, and we apply a **sigmoid** to
$C, x, y, w, h$ (so they stay in $[0, 1]$) and a **softmax** to the class scores. The original paper used a linear
output, but this makes training much more stable.

In [ ]:
def split_predictions(pred):
    """(N, S, S, B*5+K) raw -> boxes (N, S, S, B, 5) [C, x, y, w, h] and class probabilities (N, S, S, K)."""
    boxes = torch.sigmoid(pred[..., : B * 5].reshape(*pred.shape[:3], B, 5))
    class_probs = torch.softmax(pred[..., B * 5 :], dim=-1)
    return boxes, class_probs


pred = torch.randn(1, S, S, B * 5 + K)          # a fake network output
pred_boxes, class_probs = split_predictions(pred)
print("raw output:   ", tuple(pred.shape))
print("boxes:        ", tuple(pred_boxes.shape), " (N, S, S, B, [C, x, y, w, h])")
print("class probs:  ", tuple(class_probs.shape), " sum over classes =", class_probs[0, 0, 0].sum().item())

To compare predictions and targets we need all the boxes in the same reference: `cell_to_image` converts the
centre from "relative to the cell" to "relative to the image": $c_x = (\text{col} + x) / S$, $c_y = (\text{row} + y) / S$.

In [ ]:
def cell_to_image(xywh):
    """(N, S, S, n_boxes, 4) with x, y relative to the cell -> [cx, cy, w, h] relative to the image."""
    rows = torch.arange(S).view(S, 1, 1)         # broadcasting over (S, S, n_boxes)
    cols = torch.arange(S).view(1, S, 1)
    cx = (xywh[..., 0] + cols) / S
    cy = (xywh[..., 1] + rows) / S
    return torch.stack([cx, cy, xywh[..., 2], xywh[..., 3]], dim=-1)


recovered = cell_to_image(target[None, ..., 1:5].unsqueeze(-2))[0, ..., 0, :]    # (S, S, 4)
mask = target[..., 0] == 1
print("Recovered centres and sizes:\n", recovered[mask])
print("Original:\n", boxes_cxcywh)

---
## 5. The architecture

As in the slides:
1. a **backbone** that extracts features: a ResNet50 pretrained on ImageNet, without its final layers;
2. a **1×1 convolution** that reduces the number of feature maps;
3. a **fully connected** head;
4. a **reshape** from a vector to $S \times S \times (5B + K)$.

### The backbone
A ResNet ends with global average pooling and a 1000-class `fc` layer: for detection we remove them and keep the
convolutional part. Note: on a 224×224 image the ResNet reduces the resolution by 32 → a **7×7** feature map, exactly
our grid! (Here we create it with random weights, just to look at the shapes; the pretrained weights come next.)

In [ ]:
resnet = models.resnet50()
print([name for name, _ in resnet.named_children()])

backbone = nn.Sequential(*list(resnet.children())[:-2])       # drop avgpool and fc
features = backbone(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE))
print("Backbone output:", tuple(features.shape), " (batch, channels, H, W)")

### Pretrained weights
Training the backbone from scratch would require millions of images. Instead we start from weights **pretrained on
ImageNet** (1.2 million images, 1000 classes): the backbone already knows how to extract edges, textures, parts of
objects... and we only need to adapt it to detection (**transfer learning**).

`torchvision` downloads them for us: we just pass `weights=...` when creating the model. Each set of weights also
carries useful metadata: the accuracy, the class names, and the preprocessing it expects.

In [ ]:
weights = models.ResNet50_Weights.DEFAULT                  # the best ImageNet weights available
print("Weights:", weights)
print("ImageNet top-1 accuracy:", weights.meta["_metrics"]["ImageNet-1K"]["acc@1"], "%")
print("Preprocessing they expect:", weights.transforms())

resnet_pretrained = models.resnet50(weights=weights).eval()    # downloads ~100 MB the first time

# The complete pretrained network is an ImageNet classifier: let's ask it what it sees in our VOC image
with torch.no_grad():
    probs = resnet_pretrained(weights.transforms()(img).unsqueeze(0)).softmax(dim=1)[0]
top5 = probs.topk(5)
for p, idx in zip(top5.values, top5.indices):
    print(f"  {weights.meta['categories'][idx]:20s} {100 * p:.1f}%")

### The 1×1 convolution
A convolution with a 1×1 kernel does not look at the neighbours: at each position it computes a linear combination
of the channels. It is the standard way to **change the number of feature maps** cheaply. Here it reduces 2048
channels to 256, which makes the following fully connected layer 8 times smaller.

In [ ]:
reduce = nn.Conv2d(2048, 256, kernel_size=1)
reduced = reduce(features)
print(f"{tuple(features.shape)} -> {tuple(reduced.shape)}")
print(f"Parameters of the 1x1 conv: {sum(p.numel() for p in reduce.parameters()):,}")
print(f"Flattened size: 2048*7*7 = {2048 * 49:,}  ->  256*7*7 = {256 * 49:,}")

### Fully connected + reshape
The fully connected head outputs one long vector of $S \cdot S \cdot (5B + K) = 1470$ values, which we reshape into
the grid. The fully connected layer sees the **whole** image, so each cell can use global context.

In [ ]:
flat = torch.randn(4, S * S * (B * 5 + K))          # what a final nn.Linear(..., 1470) would return, batch of 4
grid = flat.view(-1, S, S, B * 5 + K)
print(f"{tuple(flat.shape)} -> {tuple(grid.shape)}")

👉 Putting these pieces together into the `YOLONet` class is **your exercise** in `yolo_voc.py`.

---
## 6. Intersection over Union (IoU)

How much do two boxes overlap? The standard measure is
$\text{IoU} = \dfrac{\text{area of intersection}}{\text{area of union}}$: 1 for identical boxes, 0 for disjoint ones.
It is used in the loss, in NMS and in the evaluation.

In [ ]:
def iou(a, b):
    """IoU between corresponding boxes, (..., 4) in xyxy."""
    x1 = torch.max(a[..., 0], b[..., 0])
    y1 = torch.max(a[..., 1], b[..., 1])
    x2 = torch.min(a[..., 2], b[..., 2])
    y2 = torch.min(a[..., 3], b[..., 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)     # 0 if they don't overlap
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (area_a + area_b - inter)


ref = torch.tensor([20., 20., 60., 60.])
others = torch.tensor([[20., 20., 60., 60.],    # identical
                       [30., 30., 70., 70.],    # shifted
                       [30., 30., 50., 50.],    # contained
                       [70., 70., 90., 90.]])   # disjoint

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, other in zip(axes, others):
    for box, color in [(ref, "tab:blue"), (other, "tab:red")]:
        ax.add_patch(plt.Rectangle(box[:2], box[2] - box[0], box[3] - box[1], fill=False, color=color, lw=2))
    ax.set_xlim(0, 100)
    ax.set_ylim(100, 0)
    ax.set_aspect("equal")
    ax.set_title(f"IoU = {iou(ref, other):.2f}")
plt.show()

print("Same as torchvision.ops.box_iou?", torch.allclose(iou(ref, others), ops.box_iou(ref[None], others)[0]))

---
## 7. The YOLO loss

The loss is a sum of squared errors with five terms. With $\mathbb{1}^{obj}_{ij} = 1$ if box $j$ of cell $i$ is
**responsible** for an object:

$$
\begin{aligned}
\mathcal{L} = \; & \lambda_{coord} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}^{obj}_{ij}
    \left[ (x_i - \hat{x}_i)^2 + (y_i - \hat{y}_i)^2 \right] && \text{centre} \\
+ \; & \lambda_{coord} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}^{obj}_{ij}
    \left[ (\sqrt{w_i} - \sqrt{\hat{w}_i})^2 + (\sqrt{h_i} - \sqrt{\hat{h}_i})^2 \right] && \text{size} \\
+ \; & \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}^{obj}_{ij} (C_i - \hat{C}_i)^2 && \text{confidence, object} \\
+ \; & \lambda_{noobj} \sum_{i=0}^{S^2} \sum_{j=0}^{B} \mathbb{1}^{noobj}_{ij} (C_i - \hat{C}_i)^2 && \text{confidence, no object} \\
+ \; & \sum_{i=0}^{S^2} \mathbb{1}^{obj}_{i} \sum_{c \in classes} (p_i(c) - \hat{p}_i(c))^2 && \text{class}
\end{aligned}
$$

Some details worth noticing:
- **Responsible box**: a cell predicts $B$ boxes but contains (at most) one object; the box with the highest IoU with
  the ground truth is responsible for it. This makes the $B$ predictors specialize (e.g. tall vs wide objects).
- **$\lambda_{coord} = 5$**: localization errors are weighted more.
- **$\lambda_{noobj} = 0.5$**: most cells contain no object; without this weight their "no object" error would
  dominate the loss and push all confidences to zero.
- **Square roots of $w, h$**: an error of 10 pixels matters much more on a small box than on a large one; the square
  root makes the loss more sensitive to errors on small boxes.
- The target confidence $C$ is 1 for the responsible box (the paper uses the IoU; 1 is a common simplification).

### Preparing the ingredients
Before computing the five terms we need the **masks** $\mathbb{1}^{obj}_i$, $\mathbb{1}^{obj}_{ij}$ and
$\mathbb{1}^{noobj}_{ij}$, and the predictions and targets in comparable shapes. Let's prepare them on the target of
our example and the random prediction of §4:

In [ ]:
tgt_batch = target.unsqueeze(0)                                  # (1, S, S, 5+K): a batch of 1 image

# 1^obj_i: the cells that contain an object
obj = tgt_batch[..., 0]                                          # (1, S, S)

# IoU between each of the B predicted boxes and the ground-truth box of the same cell (in image coordinates)
pred_xyxy = ops.box_convert(cell_to_image(pred_boxes[..., 1:5]), "cxcywh", "xyxy")               # (1, S, S, B, 4)
true_xyxy = ops.box_convert(cell_to_image(tgt_batch[..., 1:5].unsqueeze(-2)), "cxcywh", "xyxy")  # (1, S, S, 1, 4)
ious = iou(pred_xyxy, true_xyxy.expand_as(pred_xyxy))                                            # (1, S, S, B)

# 1^obj_ij: the responsible box = highest IoU, only in the cells that contain an object
resp = torch.zeros_like(ious).scatter_(-1, ious.argmax(-1, keepdim=True), 1.0) * obj.unsqueeze(-1)   # (1, S, S, B)
# 1^noobj_ij: all the other boxes
noobj = 1 - resp

# Ground-truth [x, y, w, h] with a dimension of size 1 for the boxes: it broadcasts against the B predictions
tgt = tgt_batch[..., 1:5].unsqueeze(-2)                                                          # (1, S, S, 1, 4)

for (row, col, j) in torch.nonzero(resp[0]).tolist():
    print(f"cell ({row}, {col}): IoU of the {B} boxes = {ious[0, row, col].tolist()} -> box {j} is responsible")
print(f"\nResponsible boxes: {int(resp.sum())},  boxes with no object: {int(noobj.sum())}  (out of {S * S * B})")

Now every symbol of the formula corresponds to a tensor, and all these tensors broadcast together
(in `yolo_voc.py`, `pred_boxes` is simply called `boxes`):

| Symbol | Tensor | Shape |
|---|---|---|
| $\mathbb{1}^{obj}_i$ | `obj` | `(N, S, S)` |
| $\mathbb{1}^{obj}_{ij}$ / $\mathbb{1}^{noobj}_{ij}$ | `resp` / `noobj` | `(N, S, S, B)` |
| $\hat{C}_{ij}$ (predicted) | `pred_boxes[..., 0]` | `(N, S, S, B)` |
| $\hat{x}, \hat{y}$ / $\hat{w}, \hat{h}$ (predicted) | `pred_boxes[..., 1:3]` / `pred_boxes[..., 3:5]` | `(N, S, S, B, 2)` |
| $x, y$ / $w, h$ (target) | `tgt[..., 0:2]` / `tgt[..., 2:4]` | `(N, S, S, 1, 2)` |
| $\hat{p}_i(c)$ / $p_i(c)$ | `class_probs` / `tgt_batch[..., 5:]` | `(N, S, S, K)` |

The general recipe for each term: **squared error → sum over the last dimension if needed (`.sum(-1)`) → multiply by
the right mask → sum everything (`.sum()`)**.

👉 Composing the five terms is **your exercise** in `YoloLoss.forward` of `yolo_voc.py`, where all these ingredients
are already prepared. The script checks your loss before training: a perfect prediction must give a loss of ~0.

---
## 8. From the output to the detections

At inference time the network outputs $7 \cdot 7 \cdot 2 = 98$ boxes for every image. To get the final detections:

1. compute a **class-specific score** for each box: $\text{score} = C \cdot \max_c p(c)$
   ("there is an object" × "it is of this class");
2. discard the boxes with a score below a **threshold** (e.g. 0.2);
3. apply **Non-Maximum Suppression** (NMS) to remove duplicates.

### Non-Maximum Suppression
Several boxes often detect the same object. NMS sorts the boxes by score, keeps the best one, removes all the boxes
that overlap it with IoU above a threshold (e.g. 0.5), and repeats with the remaining ones.

In [ ]:
dets = torch.tensor([[50., 40., 150., 160.],     # three detections of the same object...
                     [55., 50., 160., 170.],
                     [40., 30., 140., 150.],
                     [180., 60., 260., 150.],    # ...and two of another one
                     [185., 70., 270., 155.]])
scores = torch.tensor([0.9, 0.75, 0.6, 0.8, 0.7])

keep = ops.nms(dets, scores, iou_threshold=0.5)
print("Kept boxes:", keep.tolist())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, idx, title in [(axes[0], range(len(dets)), "before NMS"), (axes[1], keep.tolist(), "after NMS")]:
    for i in idx:
        x1, y1, x2, y2 = dets[i].tolist()
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=f"C{i}", lw=2))
        ax.text(x1, y1 - 3, f"{scores[i]:.2f}", color=f"C{i}")
    ax.set_xlim(0, 300)
    ax.set_ylim(200, 0)
    ax.set_title(title)
plt.show()

### The complete decoder
`decode` puts the three steps together. `ops.batched_nms` applies NMS separately for each class (a person and a
horse can overlap without one suppressing the other).

In [ ]:
def decode(pred, conf_thresh=0.2, nms_iou=0.5):
    boxes, class_probs = split_predictions(pred)
    xyxy = ops.box_convert(cell_to_image(boxes[..., 1:5]), "cxcywh", "xyxy").clamp(0, 1)
    cls_prob, cls_idx = class_probs.max(dim=-1)
    scores = boxes[..., 0] * cls_prob.unsqueeze(-1)                  # class-specific confidence
    labels = cls_idx.unsqueeze(-1).expand_as(scores)

    results = []
    for i in range(pred.shape[0]):
        b, s, l = xyxy[i].reshape(-1, 4), scores[i].reshape(-1), labels[i].reshape(-1)
        keep = s > conf_thresh
        b, s, l = b[keep], s[keep], l[keep]
        keep = ops.batched_nms(b, s, l, nms_iou)
        results.append((b[keep], s[keep], l[keep]))
    return results

Sanity check: let's build the output that a **perfect** network would produce for our image (inverting the sigmoid
with the `logit` function) and decode it. We must get back exactly the ground-truth boxes: this verifies that
encoding and decoding are consistent.

In [ ]:
perfect = torch.full((1, S, S, B * 5 + K), -10.0)                  # sigmoid(-10) ~ 0: no object anywhere
for row, col in torch.nonzero(target[..., 0]).tolist():
    perfect[0, row, col, 0:5] = torch.logit(target[row, col, 0:5].clamp(1e-4, 1 - 1e-4))   # box 1
    perfect[0, row, col, B * 5 :] = target[row, col, 5:] * 20                            # class scores

det_boxes, det_scores, det_labels = decode(perfect)[0]
names = [f"{VOC_CLASSES[l]} {s:.2f}" for l, s in zip(det_labels, det_scores)]
plt.figure(figsize=(5, 5))
show(draw_bounding_boxes(img224, det_boxes * IMG_SIZE, labels=names, colors="lime", width=2),
     "decoded 'perfect' prediction")
plt.show()

---
## 9. How detectors are evaluated: mAP

A detection is a **true positive** if it has the right class and an IoU ≥ 0.5 with a ground-truth box not already
matched; otherwise it is a **false positive**. Sorting the detections of a class by score, we get a
**precision–recall curve**:
- *precision*: which fraction of my detections are correct?
- *recall*: which fraction of the objects did I find?

The **Average Precision** (AP) is the area under this curve, and the **mAP@0.5** is the mean of the APs of the 20
classes. `yolo_voc.py` computes it on the test set after every epoch.

Two reference numbers:
- even the "perfect" predictions we just built do **not** reach 100% on the VOC 2007 test set, but about **95%**:
  with one object per cell, the objects whose centres fall in an already occupied cell can never be detected. This
  is the price of the $7 \times 7$ grid (later YOLO versions fix it with more boxes per cell and finer grids);
- the original YOLO v1 reached **63.4%** mAP, with a much longer training at 448×448 and extra training data.

---
## 10. Logging the training with TensorBoard

Printing the loss is fine for a few epochs, but for real trainings we want **plots that update live**, comparisons
between runs, images of the predictions... **TensorBoard** does all this. PyTorch writes the logs with a
`SummaryWriter`; TensorBoard reads them and shows them in the browser.

The main methods:

| Method | What it logs |
|---|---|
| `writer.add_scalar(tag, value, step)` | a number: loss, accuracy, learning rate... |
| `writer.add_scalars(tag, {name: value}, step)` | several numbers in the same plot |
| `writer.add_image(tag, img, step)` | an image (e.g. the predicted boxes) |
| `writer.add_histogram(tag, tensor, step)` | the distribution of a tensor (e.g. the weights) |
| `writer.add_graph(model, input)` | the computational graph of the model |

Let's log a small training: the regression on a noisy sine from Lab 1, repeated with **two learning rates** to
compare them. Each run goes into its own sub-folder of `runs/`.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

x = torch.linspace(-3, 3, 200).unsqueeze(1)
y = torch.sin(x) + 0.1 * torch.randn_like(x)

for lr in [1e-3, 1e-2]:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 1))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    writer = SummaryWriter(f"runs/demo_sine_lr{lr}")
    writer.add_graph(model, x)

    for step in range(500):
        loss = nn.functional.mse_loss(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        writer.add_scalar("loss/train", loss.item(), step)
        if step % 50 == 0:
            writer.add_histogram("weights/first_layer", model[0].weight, step)

    # a figure with the final fit
    with torch.no_grad():
        fig = plt.figure(figsize=(5, 3))
        plt.scatter(x, y, s=5)
        plt.plot(x, model(x), color="tab:red")
        plt.title(f"lr = {lr}")
    writer.add_figure("fit", fig, step)
    writer.close()
    print(f"lr={lr}: final loss {loss.item():.4f}")

And the images of a detector: we log our VOC example with its ground-truth boxes.

In [ ]:
writer = SummaryWriter("runs/demo_images")
gt = draw_bounding_boxes(img224, boxes224, labels=[VOC_CLASSES[l] for l in labels], colors="red", width=2)
writer.add_image("voc/ground_truth", gt, 0)         # expects (C, H, W)
writer.close()

### Opening TensorBoard

- **In the notebook**: run the next cell (the plots appear inline).
- **From a terminal**: `tensorboard --logdir runs` and open http://localhost:6006 in the browser.
- **In VS Code**: command palette → *Python: Launch TensorBoard*.

Things to try in the interface: the *Scalars* tab (compare the two learning rates, use the *smoothing* slider), and the
*Images*, *Graphs*, *Histograms* tabs. During the exercise, `yolo_voc.py` logs the loss, each term of the YOLO loss,
the mAP, and the predicted boxes on some test images after every epoch.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs